In [8]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "grueneisen2017children")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Grueneisen_et_al_2017_Scientific_Reports_CHRS.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [9]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="grueneisen2017children"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df.columns


In [10]:
df.rename(columns={"triainingphase": "training_phase",
    "subject": "ape",
    "stooge": "ape_2",
    "species": "species_original"}, inplace=True)

df['group'].replace('c', np.nan, inplace=True)
df = df.assign(role='focal_participant')
df = df.assign(role_2='stooge')

In [11]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)
    df['ape_2'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')

In [12]:
df['dyad']=df.ape.str.cat(df.ape_2, sep='_')


df['study'] = ''
df.loc[df.phase == 1, ['study']] = '1a'
df.loc[df.phase == 2, ['study']] = '1b'

df.rename(columns={"ape": "participant", 
                   "ape_2":"participant_2", 
                   "group":"species_subgroup",
                   'choice_subject_number':'choice_focal_participant_number',
                   'choice_subject_barrier':'choice_focal_participant_barrier',
                   'study':'experiment'}, inplace=True)
# df.columns

In [13]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')

two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [14]:
df=df[['study_id','experiment', 'year', 'month','day',  'participant','age_in_years', 'sex', 'role', 
       'participant_2', 'age_in_years_2', 'sex_2','role_2',
         'species', 'dyad','species_subgroup', 'phase', 'session', 'trial','condition',
       'barrier_position', 'choice_focal_participant_number', 'choice_focal_participant_barrier',
       'success', 'barrier_position_dummy']]


exp1 = df[df['experiment'] == '1a'] 
exp2 = df[df['experiment'] == '1b']

experiments = [[exp1, 'grueneisen2017children_exp1a'], ##connects df with name of output dataset
                [ exp2, 'grueneisen2017children_exp1b']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')## drop empty rows/columns
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

